<a href="https://colab.research.google.com/github/MiriamSchirmer/online-harm-nlp-ws2627/blob/main/notebooks/04_Automated_Fact_Checking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automated fact-checking: claim detection, evidence retrieval, verification and explanation

**Applied NLP for Online Harm Detection**, University of Konstanz, WS 2026/27

Original version of this notebook by [Dustin Wright](https://dustinbwright.com), held at a [conference tutorial](https://sites.google.com/view/ic2s2-misinformation-and-ai/startseite) led by Miriam Schirmer, Julia Mendelsohn, Dustin Wright, and Agnes Horvat.

This notebook needs a GPU: **Runtime → Change runtime type → T4 GPU → Save**.


####Fact Checking
Automated fact checking is traditionally broken down into four steps:

1. **Claim detection**: automatically identifying statements **that should be evaluated as true or false**.
2. **Evidence retrieval**: given a claim to be fact checked, **find evidence which can be used to fact check the claim**.
3. **Claim verification**: given a claim and some evidence, **predict if the claim is true, false, or that there isn't enough information to perform the fact check**.
4. **Explanation/justification**: after performing the fact check, **write an explanation which justifies the prediction**.

## Set up the environment

In [ ]:
# bm25s: BM25 search (Part 2)
!pip install -q bm25s #-q means 'quiet' - less output is printed

In [ ]:
import torch                                         # deep learning library (runs the models on the GPU)
import numpy as np                                   # 'numbers'
import pandas as pd                                  # tables (data frames)
import matplotlib.pyplot as plt                      # plots
from transformers import pipeline                    # Hugging Face: ready-to-use models for one task
from sklearn.metrics import classification_report    # precision, recall and F1 per class

print(torch.cuda.is_available())   # should print True

# PART 1: Claim Detection

<img src='https://drive.google.com/uc?id=1P9x2ydBsL5gqmEwJ9LTxE5ns7O3qGfWk'>


We will go over what data is needed to train a model and popular evaluation datasets.

We use a model that has already been trained, so we do not train it here.

### Data

We'll use the check-worthiness detection data from the [CLEF CheckThat! shared task from 2024](https://checkthat.gitlab.io/clef2024/task1/) (Task 1). The data consists of tweets and transcriptions of political speeches labeled for whether or not they are check-worthy.

The data (in multiple languages) can be found [here](https://gitlab.com/checkthat_lab/clef2024-checkthat-lab/-/tree/main/task1/data). We use the English training data (22,501 sentences) and the English dev-test data (318 sentences), saved in the course GitHub repository.

In [ ]:
train_url = "https://raw.githubusercontent.com/MiriamSchirmer/online-harm-nlp-ws2627/main/data/checkthat24_train.tsv"
test_url = "https://raw.githubusercontent.com/MiriamSchirmer/online-harm-nlp-ws2627/main/data/checkthat24_test.tsv"

# the files are tab-separated (.tsv), so we set sep="\t"
train_data = pd.read_csv(train_url, sep="\t")
test_data = pd.read_csv(test_url, sep="\t")
train_data.head()

### Let's look at a couple of samples

In [ ]:
print("NOT Check-Worthy")
print(train_data.iloc[0]["Text"])
print()
print("Check-Worthy")
print(train_data.iloc[3]["Text"])

## 💬 Discuss: What makes a claim check-worthy?

### Model

We use a model that was fine-tuned for check-worthiness detection on the training data: [`dwright37/icwsm_cw_base`](https://huggingface.co/dwright37/icwsm_cw_base). It is based on [ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large), a modern version of the BERT architecture. The download is about 1.6 GB.

1. the fine-tuned model,
2. the tokenizer of ModernBERT-large (the fine-tuned model has none of its own),
3. put the model on the GPU.

In [ ]:
cw_classifier = pipeline(
    "text-classification",
    model="dwright37/icwsm_cw_base",            # 1.
    tokenizer="answerdotai/ModernBERT-large",   # 2.
    device_map="auto",                          # 3.
)

# predict a label for each test sentence, 32 sentences at a time
predictions = cw_classifier(test_data["Text"].tolist(), batch_size=32)
predictions[:5]

### Evaluation method

We will evaluate the performance of the model using binary F1 score (the row `Yes`).

The model returns `LABEL_0` (not check-worthy, `No`) and `LABEL_1` (check-worthy, `Yes`).

In [ ]:
label_map = {"LABEL_0": "No", "LABEL_1": "Yes"}
test_data["pred_base"] = [label_map[p["label"]] for p in predictions]

print(classification_report(test_data["class_label"], test_data["pred_base"], digits=3))

When the model predicts a claim is check-worthy, oftentimes this is correct. However, the model misses *many* check-worthy claims. Why is this?

One thing to note about check-worthiness data: there tends to be strong label imbalance.

In [ ]:
# plot the label distribution
train_data["class_label"].value_counts().plot(kind="barh", color="grey")
plt.show()

## Accounting for label imbalance

We can improve performance by *a lot* if we take this label imbalance into account.

The model [`dwright37/icwsm_cw_weighted`](https://huggingface.co/dwright37/icwsm_cw_weighted) was trained in the same way, but with a weighted cross entropy loss. Each class gets the weight: number of sentences / (2 × number of sentences in this class), so mistakes on the rare check-worthy class cost more.

In [ ]:
# free the GPU memory of the first model
del cw_classifier
torch.cuda.empty_cache()

cw_classifier = pipeline(
    "text-classification",
    model="dwright37/icwsm_cw_weighted",
    tokenizer="answerdotai/ModernBERT-large",
    device_map="auto",
)

predictions = cw_classifier(test_data["Text"].tolist(), batch_size=32)
test_data["pred_weighted"] = [label_map[p["label"]] for p in predictions]

print(classification_report(test_data["class_label"], test_data["pred_weighted"], digits=3))

**The improvement is mainly driven by a large improvement in recall, without losing much precision.**

### ✏️ Your turn

Write your own sentences and let the model decide whether they are check-worthy. Try a claim with a number, an opinion and a question.

In [ ]:
my_sentences = [
    "Unemployment has doubled since the last election.",
    "I think this is the best city in the world.",
    "Who is going to pay for all of this?",
]

for sentence, p in zip(my_sentences, cw_classifier(my_sentences)):
    print(label_map[p["label"]], round(p["score"], 2), "|", sentence)

In [ ]:
# free the GPU memory before Part 2
del cw_classifier
torch.cuda.empty_cache()

# PART 2: Evidence Retrieval

<img src='https://drive.google.com/uc?id=1ar3ukXjvgfngEZy4Z2J1cM-2t_F2TF8U'>


Evidence retrieval involves the following:

- Create a database of potential evidence to search over
- Vectorize the evidence for fast retrieval
- Given a claim, vectorize the claim
- Rank each piece of evidence based on its similarity to the claim, using some vector similarity metric

We will look at two ways to vectorize our text: BM25 (Best Matching 25, a ranking algorithm) and Sentence Transformers.

## 💬 Discuss: What data would you use as an evidence corpus?

### Dataset

For the rest of the tutorial we will rely on the [FEVER dataset](https://arxiv.org/abs/1803.05355), which contains fact check-worthy claims paired with factuality labels. The evidence set comes from a large Wikipedia dump.

We use the version [copenlu/fever_gold_evidence](https://huggingface.co/datasets/copenlu/fever_gold_evidence), which has gold evidence paired with the claims needed to fact check them.

Note: we only use its test split (16,039 claims), saved in the course GitHub repository. The original tutorial uses all three splits (260,251 claims).

In [ ]:
fever_url = "https://raw.githubusercontent.com/MiriamSchirmer/online-harm-nlp-ws2627/main/data/fever_test.jsonl"

fever = pd.read_json(fever_url, lines=True)   # .jsonl: one claim per line
fever.head()

The `evidence` column contains lists of evidence snippets, including the title of the original Wikipedia article, sentence number, and evidence text.

In [ ]:
fever["evidence"][24]

Let's create a flat database of evidence texts to use for retrieval:

1. `corpus`: every evidence text, each only once,
2. `claim_to_gold`: for each claim, its gold evidence texts.

In [ ]:
corpus = []
claim_to_gold = {}

for claim, evidence_list in zip(fever["claim"], fever["evidence"]):
    for title, sentence_number, sentence in evidence_list:
        corpus.append(sentence)                                  # 1.
        claim_to_gold.setdefault(claim, []).append(sentence)     # 2.

corpus = list(dict.fromkeys(corpus))   # remove duplicates, keep the order
print(len(corpus), "pieces of evidence")

We will test two different ways to retrieve evidence: BM25 and Sentence Transformers.

<img src='https://drive.google.com/uc?id=1PDMDkVt5aeZUf7h81I50h7wqM4dZfGpq'>

Both methods are based on a similar concept: turn each piece of text (claims and evidence) into **vectors**, and rank each piece of evidence based on its vector similarity to a given claim.

The difference between the two methods lies in *how* they turn text into vectors.

### BM25

BM25 is a **bag-of-words** retrieval method.

This means that it is based on the relative frequencies of different words appearing in each piece of text.

At a *very* high level, it works as follows:

You start with a corpus of text, from which a vocabulary $V$ is constructed, and the statistics of each word in the vocabulary are calculated.

Then, a given piece of text is turned into a vector of size $V$, where each index is the index of a word in the vocabulary. The value at each index is calculated based on the frequency of each word in the supplied text relative to the frequency of that word in the original corpus.

Note: this is close to TF-IDF. Like TF-IDF, BM25 gives rare words more weight. In addition, a word that occurs many times counts less and less (parameter `k1`), and long texts do not win just because they are long (parameter `b`). **BM25 is made for ranking texts for a search query, not as features for a classifier.**

In [ ]:
import bm25s   # BM25 search

topk = 10   # number of results per claim

# First, index the corpus of evidence
retriever = bm25s.BM25(corpus=corpus)
retriever.index(bm25s.tokenize(corpus))

In [ ]:
# Now we can try to do some retrieval
claim = fever["claim"][5]

print("CLAIM")
print("=====")
print(claim)
print()

# Retrieve the top 10 results
results, scores = retriever.retrieve(bm25s.tokenize(claim), k=topk)

print("EVIDENCE")
print("=====")
for sentence in results[0]:
    print(sentence)
    print()

### Sentence Transformers

<img src='https://drive.google.com/uc?id=1nZouGErNgE4uHlbJ3m1Hup0VbaKd75-L'/>


Sentence transformers turn pieces of text into vectors based on their *semantics*. In other words, pieces of text which convey the same meaning will result in vectors which are close to each other, while those with dissimilar meaning will be far away. These vectors are also referred to as **embeddings**.

Sentence transformers produce embeddings by using BERT as a feature extractor. The base BERT model is fine-tuned by comparing billions of pairs of text and optimizing the base BERT network with an objective function which causes the features of similar pieces of text to be close in the feature space, and different pieces of text to be far away in the embedding space.

For more details, please see [this paper](https://arxiv.org/pdf/1908.10084) or read [the documentation](https://sbert.net/).

In [ ]:
from sentence_transformers import SentenceTransformer   # loads embedding models from Hugging Face

# Load the pre-trained model (the same as in Session 5)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings of the evidence
evidence_embeddings = embedding_model.encode(corpus, show_progress_bar=True, normalize_embeddings=True)

1. Get an embedding of the claim.
2. Measure the cosine similarity with the evidence corpus. The embeddings are normalised, so this is the dot product (`@`), as in Session 5.
3. Get the top 10 pieces of evidence.

In [ ]:
claim_embedding = embedding_model.encode([claim], normalize_embeddings=True)[0]   # 1.
similarities = evidence_embeddings @ claim_embedding                              # 2.
top_indices = np.argsort(similarities)[::-1][:topk]                               # 3.

print("CLAIM")
print("=====")
print(claim)
print()

print("EVIDENCE")
print("=====")
for i in top_indices:
    print(corpus[i])
    print()

### Evaluation

How do we evaluate if our evidence retrieval system is working well?

The output of our retrieval system will be a **ranked list** of pieces of evidence. When we have access to **gold evidence** (i.e., evidence that we know is relevant for the fact check), we can evaluate our system based on how highly the relevant evidence is ranked.

There are many metrics one can use for this; in this tutorial we will use Mean Reciprocal Rank (MRR).

For each claim, we take 1 divided by the rank of the first relevant piece of evidence (1/1 if it is first, 1/2 if it is second, and so on; 0 if it is not in the top 10). MRR is the mean over all claims.

In [ ]:
def reciprocal_rank(claim, retrieved):
    for rank, sentence in enumerate(retrieved, start=1):
        if sentence in claim_to_gold[claim]:
            return 1 / rank
    return 0   # no relevant evidence in the top 10

### Compare BM25 and Sentence Transformers on the retrieval task

In [ ]:
# Get BM25 results for the first 1000 claims
claims = list(claim_to_gold.keys())[:1000]
results, scores = retriever.retrieve(bm25s.tokenize(claims), k=topk)

reciprocal_ranks = [reciprocal_rank(claim, retrieved) for claim, retrieved in zip(claims, results)]
print(f"BM25 MRR: {np.mean(reciprocal_ranks):.3f}")

In [ ]:
# Get transformer results for the first 1000 claims
claim_embeddings = embedding_model.encode(claims, normalize_embeddings=True)
similarities = claim_embeddings @ evidence_embeddings.T             # one row per claim
top_indices = np.argsort(similarities, axis=1)[:, ::-1][:, :topk]   # top 10 per row

reciprocal_ranks = []
for claim, indices in zip(claims, top_indices):
    retrieved = [corpus[i] for i in indices]
    reciprocal_ranks.append(reciprocal_rank(claim, retrieved))

print(f"Transformers MRR: {np.mean(reciprocal_ranks):.3f}")

### ✏️ Your turn

Rewrite a FEVER claim in your own words, with words that are not in the evidence, and compare the top 3 results of BM25 and the sentence transformer. Which method finds the right evidence?

In [ ]:
print(claims[10])   # ✏️ a FEVER claim (change the number for another one)
print(claim_to_gold[claims[10]][0])   # its gold evidence
print()

my_claim = "Write the claim in your own words here."

print("BM25")
results_bm25, scores_bm25 = retriever.retrieve(bm25s.tokenize(my_claim), k=3)
for sentence in results_bm25[0]:
    print("-", sentence)

print()
print("SENTENCE TRANSFORMER")
my_embedding = embedding_model.encode([my_claim], normalize_embeddings=True)[0]
for i in np.argsort(evidence_embeddings @ my_embedding)[::-1][:3]:
    print("-", corpus[i])

# PART 3: Claim Verification

<img src='https://drive.google.com/uc?id=1swWucmrV8NywVPiDEDYjdpa5TEQG5Kba'>


The third step in automatic fact checking is claim verification.

Claim verification involves two steps:

1. Given a claim, retrieve evidence which either supports or refutes the claim.
2. Determine if the evidence SUPPORTS or REFUTES the claim, or if there is NOT ENOUGH INFORMATION.

In [ ]:
# all evidence texts of a claim in one string (the text is the 3rd part of each piece, index 2)
fever["evidence_text"] = fever["evidence"].apply(lambda evidence_list: "\n".join(e[2] for e in evidence_list))

# Show examples of claims and their labels
for k in [0, 14, 17]:
    print(f"Claim: {fever['claim'][k]}")
    print(f"Label: {fever['label'][k]}")
    print(f"Evidence: {fever['evidence_text'][k]}")
    print()

We'll use a pre-trained model for this and compare to using a large language model (LLM).

### Natural language inference (NLI)

NLI is a classification task with two texts as input: a **premise** and a **hypothesis**. The model predicts whether the hypothesis follows from the premise:

- **entailment**: the hypothesis follows from the premise.
- **contradiction**: the premise shows that the hypothesis is false.
- **neutral**: the premise says nothing about it either way.

Example: premise *"The concert was cancelled because of the storm."* Hypothesis *"The concert took place."* → contradiction.

**For fact checking, the evidence is the premise and the claim is the hypothesis.**

## Input

The input data to the pipeline is a list of dictionaries.

Each dictionary has two items:

`text`: The evidence that we are using for the fact check.

`text_pair`: The claim that we are fact checking.

The pipeline will then return a list of predictions, where each prediction has a label and a score (probability of that label).

The model that we are using ([`ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli`](https://huggingface.co/ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli)) has been trained on many different datasets for the more general task of natural language inference, which tries to predict if the `text_pair` sentence is absolutely and fully entailed by the `text`.

Therefore, we will need to map the prediction labels to the dataset labels: `entailment` to `SUPPORTS`, `contradiction` to `REFUTES` and `neutral` to `NOT ENOUGH INFO`.

In [ ]:
N = 1000   # number of test claims

# Create the pipeline using a good pretrained model
inference_pipeline = pipeline("text-classification",
                              model="ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli",
                              device_map="auto")

# the first N claims, with their evidence
verify_df = fever.head(N).copy()

inference_data = []
for claim, evidence in zip(verify_df["claim"], verify_df["evidence_text"]):
    inference_data.append({"text": evidence, "text_pair": claim})

# Run the data through the pipeline (truncation=True cuts texts that are longer than the model can read)
predictions = inference_pipeline(inference_data, batch_size=16, truncation=True)
predictions[:10]

In [ ]:
# We have to map them to the same label space
pipeline_label_map = {"entailment": "SUPPORTS", "contradiction": "REFUTES", "neutral": "NOT ENOUGH INFO"}
verify_df["nli_label"] = [pipeline_label_map[p["label"]] for p in predictions]

# Evaluate with accuracy
accuracy = (verify_df["label"] == verify_df["nli_label"]).mean()
print(f"Accuracy: {accuracy:.1%}")
print()
print(pd.crosstab(verify_df["label"], verify_df["nli_label"], rownames=["FEVER"], colnames=["model"]))

### ✏️ Your turn

**Easy:** Write your own evidence–claim pairs, for example with the concert sentence from above. Does the model get them right? Try a claim that is only partly supported.

In [ ]:
my_pairs = [
    {"text": "The concert was cancelled because of the storm.", "text_pair": "The concert took place."},
    {"text": "The concert was cancelled because of the storm.", "text_pair": "There was a storm."},
    {"text": "The concert was cancelled because of the storm.", "text_pair": "The band was paid anyway."},
]

for pair, p in zip(my_pairs, inference_pipeline(my_pairs)):
    print(p["label"], round(p["score"], 2), "|", pair["text_pair"])

## 💬 Discuss: Would you use this system in practice? How accurate should a fact verification system be?

# EXTRA (if time): LLMs

### Claim verification with large language models

We can also perform the claim verification task with an LLM.

Here we will use Qwen2.5-1.5B-Instruct (as in Session 4) for claim verification.

We use prompts from this recent paper at the FEVER workshop: https://github.com/ronit-singhal/evidence-backed-fact-checking-using-rag-and-few-shot-in-context-learning-with-llms

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM   # Hugging Face: load an LLM and its tokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"       # default: small and fast
# model_name = "Qwen/Qwen3-4B-Instruct-2507"   # larger (and usually better), needs the T4 GPU

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",        # use the precision the model was saved in (16 bit)
    device_map="auto",   # put the model on the GPU
)

def ask(prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]   # keep only the answer, not the prompt
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Model loaded on:", model.device)

### Zero-shot setup

In this example, we will simply prompt Qwen with the claim and evidence statements, with some basic instructions about how to label each example.

In [ ]:
# Set up the prompt for fact checking
ZERO_SHOT_PROMPT_TEMPLATE = """
You are tasked with classifying a given claim based on provided statements into one of the following three categories:
1. SUPPORTS: If there is sufficient evidence indicating that the claim is legitimate, classify it as SUPPORTS.
2. REFUTES: If there is any evidence contradicting the claim, classify it as REFUTES.
3. NOT ENOUGH INFO: If you cannot find any conclusive factual evidence either supporting or refuting the claim, classify it as NOT ENOUGH INFO.
This means the available information is insufficient to make a definitive judgment.

Claim: {claim}
Statements: {statements}

Pick only one from ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']
Do not print anything other than the final classification.
"""

print(ZERO_SHOT_PROMPT_TEMPLATE.format(claim=fever["claim"][0], statements=fever["evidence_text"][0]))

In [ ]:
from tqdm import tqdm   # progress bar

N_LLM = 30   # 30 keeps it to about a minute (performance metrics might not be interpretable!)

predictions = []
for claim, evidence in tqdm(zip(verify_df["claim"][:N_LLM], verify_df["evidence_text"][:N_LLM]), total=N_LLM):
    prompt = ZERO_SHOT_PROMPT_TEMPLATE.format(claim=claim, statements=evidence)
    answer = ask(prompt, max_new_tokens=10).strip().upper()
    # if the answer is not one of our labels, set it to NOT ENOUGH INFO
    if answer not in ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]:
        answer = "NOT ENOUGH INFO"
    predictions.append(answer)

llm_df = verify_df.head(N_LLM).copy()
llm_df["qwen_label"] = predictions

# Qwen and the NLI model on the same claims
print(f"Zero-shot LLM accuracy: {(llm_df['label'] == llm_df['qwen_label']).mean():.1%}")
print(f"NLI model accuracy:     {(llm_df['label'] == llm_df['nli_label']).mean():.1%}")

# PART 4: Justification/Explanation

<img src='https://drive.google.com/uc?id=1rIclhuZa4ISAiZ-2waYcITvfNsQekVMW'>


Finally, after predicting if a claim is true or false, we need to provide an explanation/justification for *why* this is the case.

For this we will turn to Qwen again and prompt it to generate an explanation for a few examples.

In [ ]:
# Set up the prompt for explanation generation
EXPLANATION_TEMPLATE = """
You are tasked with creating an explanation of a fact check for a given claim. You are provided with the original claim, the
label given to that claim ('SUPPORTS', 'REFUTES', or 'NOT ENOUGH INFO'), and the statements used to perform the fact check.
Please provide a brief explanation as to why the label is correct based on the provided evidence.

Claim: {claim}
Label: {label}
Statements: {statements}
"""

# a NOT ENOUGH INFO, a REFUTES and a SUPPORTS example
for idx in [0, 2, 24]:
    claim = fever["claim"][idx]
    evidence = fever["evidence_text"][idx]
    label = fever["label"][idx]

    prompt = EXPLANATION_TEMPLATE.format(claim=claim, label=label, statements=evidence)
    explanation = ask(prompt, max_new_tokens=200)

    print(f"Claim: {claim}")
    print(f"Evidence: {evidence}")
    print(f"Label: {label}")
    print(f"Explanation: {explanation}")
    print("=================")